In [0]:
from pyspark.sql.functions import col, when, lit, current_timestamp, count, sum as spark_sum, unix_timestamp

bronze_df = spark.read.table("nycyellow.bronze.nyctripdata_raw")

# Tag every row with which check(s) it fails, rather than just dropping immediately
checked_df = (bronze_df
    .withColumn("dq_null_check", when(col("tpep_pickup_datetime").isNull() |
                                        col("tpep_dropoff_datetime").isNull() |
                                        col("PULocationID").isNull() |
                                        col("DOLocationID").isNull() |
                                        col("fare_amount").isNull(), True).otherwise(False))
    .withColumn("dq_negative_fare", col('fare') < 0)
    .withColumn("dq_invalid_distance", col('distance') <= 0)
    .withColumn("dq_dropoff_before_pickup",
        when(col("tpep_dropoff_datetime") < col("tpep_pickup_datetime"), True).otherwise(False))
    .withColumn("dq_unrealistic_duration",
        when(
            (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) > 86400,
            True).otherwise(False))
    .withColumn("dq_invalid_passenger_count", col('passenger_count') <= 0)
)

# A row is "clean" only if it fails none of the checks
clean_df = (checked_df
    .withColumn("dq_failed_any",
        col("dq_null_check") | col("dq_negative_fare") | col("dq_invalid_distance") |
        col("dq_dropoff_before_pickup") | col("dq_unrealistic_duration") | col("dq_invalid_passenger_count"))
)

# Split: passed rows move to silver, failed rows get logged for visibility
passed_df = clean_df.filter(col("dq_failed_any") == False).drop(
    "dq_null_check", "dq_negative_fare", "dq_invalid_distance",
    "dq_dropoff_before_pickup", "dq_unrealistic_duration", "dq_invalid_passenger_count", "dq_failed_any"
)

failed_df = clean_df.filter(col("dq_failed_any") == True)

# Log a summary of what failed and how often — this is what makes your DQ dashboard objective real
dq_summary = (checked_df.select(
        spark_sum(col("dq_null_check").cast("int")).alias("null_check_failures"),
        spark_sum(col("dq_negative_fare").cast("int")).alias("negative_fare_failures"),
        spark_sum(col("dq_invalid_distance").cast("int")).alias("invalid_distance_failures"),
        spark_sum(col("dq_dropoff_before_pickup").cast("int")).alias("dropoff_before_pickup_failures"),
        spark_sum(col("dq_unrealistic_duration").cast("int")).alias("unrealistic_duration_failures"),
        spark_sum(col("dq_invalid_passenger_count").cast("int")).alias("invalid_passenger_count_failures"),
        count("*").alias("total_rows_checked"))
    .withColumn("check_run_time", current_timestamp())
)

# Write the clean rows forward for Silver to pick up
passed_df.write.format("delta").mode("append").saveAsTable("nycyellow.bronze.validated_tripdata")

# Log failed rows separately — useful for debugging and for showing your trainer what got caught
failed_df.write.format("delta").mode("append").saveAsTable("nycyellow.bronze.dq_rejected_rows")

# Append this run's summary to your metrics table — this powers the DQ dashboard objective
dq_summary.write.format("delta").mode("append").saveAsTable("nycyellow.gold.dq_metrics")